# **PARTE 1**

**Montar Drive y fijar carpeta base**

In [1]:
!pip install unidecode

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os, io
import pandas as pd
import numpy as np
from unidecode import unidecode

pd.set_option('display.max_columns', 150)
pd.set_option('display.width', 160)

BASE = r'C:\Users\leoro\Downloads\TallerIntegradorPruebas'
os.listdir(BASE)


['BD zona.csv',
 'intermediate',
 'maestra.csv',
 'Resumen_Valores-checkpoint.ipynb',
 'Resumen_Valores-VENTA_POR_FAMILIA2.csv',
 'Resumen_Valores-VENTA_POR_PRODUCTO2.csv']

 **Cargar BD zona.csv**

In [3]:
bd = pd.read_csv(f'{BASE}/BD zona.csv', encoding='latin-1', delimiter=';')
bd.head(10)

,Vendedor,Nombre Cliente,Producto,MES NUM,Mes,2025,CANTIDAD
0,Pharma - N1,ADMINISTRADORA CLINICA TRESA S.A,AGGLAD,1,ENE,0,0
1,Pharma - N1,ADMINISTRADORA CLINICA TRESA S.A,FLUMETOL NF,1,ENE,0,0
2,Pharma - N1,ADMINISTRADORA CLINICA TRESA S.A,GAAP,1,ENE,0,0
3,Pharma - N1,ADMINISTRADORA CLINICA TRESA S.A,LAGRICEL,1,ENE,0,0
4,Pharma - N1,"BENEL PEREZ,DENNY JAVIER",FLUMETOL NF,1,ENE,898.37,20
5,Pharma - N1,"BENEL PEREZ,DENNY JAVIER",TRAZIDEX U,1,ENE,1028.61,20
6,Pharma - N1,BM CLINICA DE OJOS S.A.C.,FLUMETOL NF,1,ENE,612.53,15
7,Pharma - N1,BM CLINICA DE OJOS S.A.C.,GAAP,1,ENE,1925.85,25
8,Pharma - N1,BM CLINICA DE OJOS S.A.C.,TRAZIDEX O,1,ENE,250.05,6
9,Pharma - N1,BM CLINICA DE OJOS S.A.C.,TRAZIDEX U,1,ENE,635.59,0


In [4]:
bd.columns.tolist(), bd.shape, bd.dtypes

(['Vendedor',
  'Nombre Cliente',
  'Producto',
  'MES NUM',
  'Mes',
  '2025',
  'CANTIDAD'],
 (4884, 7),
 Vendedor            str
 Nombre Cliente      str
 Producto            str
 MES NUM           int64
 Mes                 str
 2025                str
 CANTIDAD            str
 dtype: object)

In [5]:
# Renombrar nombres
bd = bd.rename(columns={
    'Vendedor':'vendedor',
    'Nombre Cliente':'cliente',
    'Producto':'producto',
    'MES NUM':'mes_num',
    'Mes':'mes_abbr',
    '2025':'monto_cancelado',
    'CANTIDAD':'cantidad'
})

bd['mes_num'] = pd.to_numeric(bd['mes_num'], errors='coerce').astype('Int64')
bd['cantidad'] = pd.to_numeric(bd['cantidad'], errors='coerce').fillna(0).astype(int)
bd['monto_cancelado'] = pd.to_numeric(bd['monto_cancelado'], errors='coerce').fillna(0.0)

for c in ['vendedor','cliente','producto','mes_abbr']:
    bd[c] = bd[c].astype(str).str.strip().str.upper()

bd.info()
print("\n")
bd.head(10)



<class 'pandas.DataFrame'>
RangeIndex: 4884 entries, 0 to 4883
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   vendedor         4884 non-null   str    
 1   cliente          4884 non-null   str    
 2   producto         4884 non-null   str    
 3   mes_num          4884 non-null   Int64  
 4   mes_abbr         4884 non-null   str    
 5   monto_cancelado  4884 non-null   float64
 6   cantidad         4884 non-null   int64  
dtypes: Int64(1), float64(1), int64(1), str(4)
memory usage: 272.0 KB




,vendedor,cliente,producto,mes_num,mes_abbr,monto_cancelado,cantidad
0,PHARMA - N1,ADMINISTRADORA CLINICA TRESA S.A,AGGLAD,1,ENE,0.00,0
1,PHARMA - N1,ADMINISTRADORA CLINICA TRESA S.A,FLUMETOL NF,1,ENE,0.00,0
2,PHARMA - N1,ADMINISTRADORA CLINICA TRESA S.A,GAAP,1,ENE,0.00,0
3,PHARMA - N1,ADMINISTRADORA CLINICA TRESA S.A,LAGRICEL,1,ENE,0.00,0
4,PHARMA - N1,"BENEL PEREZ,DENNY JAVIER",FLUMETOL NF,1,ENE,898.37,20
5,PHARMA - N1,"BENEL PEREZ,DENNY JAVIER",TRAZIDEX U,1,ENE,1028.61,20
6,PHARMA - N1,BM CLINICA DE OJOS S.A.C.,FLUMETOL NF,1,ENE,612.53,15
7,PHARMA - N1,BM CLINICA DE OJOS S.A.C.,GAAP,1,ENE,1925.85,25
8,PHARMA - N1,BM CLINICA DE OJOS S.A.C.,TRAZIDEX O,1,ENE,250.05,6
9,PHARMA - N1,BM CLINICA DE OJOS S.A.C.,TRAZIDEX U,1,ENE,635.59,0


In [6]:
bd['zona_code'] = bd['vendedor'].str.split('-').str[-1].str.strip().str.split().str[-1]

ZMAP = {
    'Z1':'LIMA', 'Z2':'CALLAO', 'Z3':'NORTE CHICO',
    'N1':'CHICLAYO', 'N2':'TRUJILLO', 'S':'CADENAS'
}
bd['zona'] = bd['zona_code'].map(ZMAP).fillna(bd['zona_code'])

bd[['vendedor','zona_code','zona']].drop_duplicates().sort_values('zona_code').head(20)


,vendedor,zona_code,zona
0,PHARMA - N1,N1,CHICLAYO
104,PHARMA - N2,N2,TRUJILLO
184,PHARMA - S,S,CADENAS
262,PHARMA - Z1,Z1,LIMA
501,PHARMA - Z2,Z2,CALLAO
673,PHARMA - Z3,Z3,NORTE CHICO


 **Banderas de negocio**

1) ¿Qué significa cada bandera?

* is_purchase → COMPRA / ENTREGA ese mes.
Se activa cuando cantidad > 0. Es la señal positiva principal para recomendaciones.

* is_payment_only → SOLO PAGO (se cobró algo de una entrega anterior).
Se activa cuando monto_cancelado > 0 y cantidad = 0. No es una compra nueva.

* is_return → DEVOLUCIÓN / AJUSTE (no hubo compra ni cobro).
Se activa cuando monto_cancelado = 0 y cantidad = 0.

* is_credit_delivery → COMPRA A CRÉDITO (se entregó, pero no se pagó ese mes).
Se activa cuando monto_cancelado = 0 y cantidad > 0. Sigue siendo compra.

En resumen de uso para el modelo:

* Entrenamos con filas is_purchase == True (incluye también los casos de crédito, porque hay entrega).

* is_payment_only y is_return no son compras nuevas; sirven como contexto, no como eventos de compra.

In [7]:
bd['is_return']          = (bd['monto_cancelado']==0) & (bd['cantidad']==0)   # devolución/ajuste
bd['is_payment_only']    = (bd['monto_cancelado']>0)  & (bd['cantidad']==0)   # solo pago (entrega pasada)
bd['is_purchase']        = (bd['cantidad']>0)                                  # compra/entrega
bd['is_credit_delivery'] = (bd['monto_cancelado']==0) & (bd['cantidad']>0)    # compra a crédito

print("Filas totales:", len(bd))
print("Compras (cantidad>0):", bd['is_purchase'].sum())
print("Solo pago:", bd['is_payment_only'].sum())
print("Devoluciones:", bd['is_return'].sum())
print("Crédito (entrega sin pago):", bd['is_credit_delivery'].sum())


Filas totales: 4884
Compras (cantidad>0): 4717
Solo pago: 109
Devoluciones: 29
Crédito (entrega sin pago): 986


In [8]:
bd.head(10)

,vendedor,cliente,producto,mes_num,mes_abbr,monto_cancelado,cantidad,zona_code,zona,is_return,is_payment_only,is_purchase,is_credit_delivery
0,PHARMA - N1,ADMINISTRADORA CLINICA TRESA S.A,AGGLAD,1,ENE,0.00,0,N1,CHICLAYO,True,False,False,False
1,PHARMA - N1,ADMINISTRADORA CLINICA TRESA S.A,FLUMETOL NF,1,ENE,0.00,0,N1,CHICLAYO,True,False,False,False
2,PHARMA - N1,ADMINISTRADORA CLINICA TRESA S.A,GAAP,1,ENE,0.00,0,N1,CHICLAYO,True,False,False,False
3,PHARMA - N1,ADMINISTRADORA CLINICA TRESA S.A,LAGRICEL,1,ENE,0.00,0,N1,CHICLAYO,True,False,False,False
4,PHARMA - N1,"BENEL PEREZ,DENNY JAVIER",FLUMETOL NF,1,ENE,898.37,20,N1,CHICLAYO,False,False,True,False
5,PHARMA - N1,"BENEL PEREZ,DENNY JAVIER",TRAZIDEX U,1,ENE,1028.61,20,N1,CHICLAYO,False,False,True,False
6,PHARMA - N1,BM CLINICA DE OJOS S.A.C.,FLUMETOL NF,1,ENE,612.53,15,N1,CHICLAYO,False,False,True,False
7,PHARMA - N1,BM CLINICA DE OJOS S.A.C.,GAAP,1,ENE,1925.85,25,N1,CHICLAYO,False,False,True,False
8,PHARMA - N1,BM CLINICA DE OJOS S.A.C.,TRAZIDEX O,1,ENE,250.05,6,N1,CHICLAYO,False,False,True,False
9,PHARMA - N1,BM CLINICA DE OJOS S.A.C.,TRAZIDEX U,1,ENE,635.59,0,N1,CHICLAYO,False,True,False,False


 **Filtrar compras**

In [9]:
compras = bd[bd['is_purchase']].copy()
# ¿Meses válidos?
compras['mes_num'].value_counts(dropna=False).sort_index().head(20)


mes_num
1    723
2    770
3    765
4    808
5    710
6    776
7    165
Name: count, dtype: Int64

* is_purchase lo creaste en el paso 5 y significa “hubo entrega” (cantidad > 0).

Con esto sacamos de la tabla:

* filas de solo pago (monto_cancelado > 0 y cantidad = 0) → no son compras nuevas, no sirven para aprender “qué viene después”.

* devoluciones/ajustes (monto_cancelado = 0 y cantidad = 0) → tampoco son compras.

.copy() es solo para trabajar en una copia y evitar advertencias de pandas.

¿Por qué filtramos? Porque para entrenar la RNN de recomendaciones, la señal que nos interesa es qué productos compró el cliente en cada mes (entrega), no si solo pagó algo pasado ni si hubo devolución.

In [10]:
# Top productos y top clientes (sirve para validar que se vea razonable)
compras['producto'].value_counts().head(15)


producto
FLUMETOL NF    570
LAGRICEL PF    541
TRAZIDEX U     520
AGGLAD         404
TRAZIDEX O     400
ELIPTIC PF     357
SOPHIPREN      325
LAGRICEL       309
GAAP PF        279
AQUADRAN       274
ZEBESTEN       196
GAAP           195
ELAR-B         117
ELIPTIC         81
LANDAX          80
Name: count, dtype: int64

Te muestra los 15 productos con más compras.

Sirve para validar que los nombres están bien y ver si hay variantes del mismo producto (errores de tipeo o espacios).

In [11]:
compras['cliente'].value_counts().head(15)


cliente
MACULA D & T S.R.L.                                                  72
INSTITUTOS OFTALMOLOGICOS ESPECIALIZADOS DR.CARLOS WONG CAM S.A.C    70
SISTEMAS DE ADMINISTRACION HOSPITALARIA S.A.C.                       69
ASOCIACION PERUANO JAPONESA                                          69
CLINICA DE OJOS DÏOPELUCE SAC                                        66
VISUAL CENTER S.A.C.                                                 65
OFTALMICA S.A.C.                                                     62
TG LASER OFTALMICA SA                                                62
SERVICIOS DE FARMACIA OFTALMOLOGICA YAHUI EIRL                       62
OPTIMA VISION SRL                                                    54
OFTALMOVISION S.A.C.                                                 53
ARBRAYSS LASER S.R.L                                                 53
CLINICA INTERNACIONAL S A                                            53
CLINICA AREQUIPA SA                                     

Muestra los 15 clientes con más compras.

Sirve para detectar outliers (un cliente con miles de compras) o nombres duplicados (mismo cliente con dos formas de escritura).

**Ordena por tiempo y mira secuencias por cliente**
Esto es lo que alimentará a la RNN (GRU).

In [12]:
compras = compras.sort_values(['cliente','mes_num','producto'])
seq_por_cliente = compras.groupby('cliente')['producto'].apply(list).reset_index()
seq_por_cliente = seq_por_cliente.rename(columns={'producto':'secuencia_productos'})

seq_por_cliente.head(10)


,cliente,secuencia_productos
0,A & E SERVICIOS MEDICOS S.A.C.,"[GAAP PF, LAGRICEL PF, SOPHIPREN, ZEBESTEN, LA..."
1,AAA SAC,"[DUSTALOX, ELIPTIC PF, TRAZIDEX U, ZEBESTEN, E..."
2,ADMINISTRADORA CLINICA RICARDO PALMA S.A.,"[AGGLAD, ELIPTIC, FLUMETOL NF, LAGRICEL, SOPHI..."
3,ADMINISTRADORA CLINICA TRESA S.A,"[AGGLAD, FLUMETOL NF, GAAP, LAGRICEL, TRAZIDEX..."
4,AGUIRRE ROCCA CESAR JOAQUIN,[TRAZIDEX O]
5,AJALCRIYA PASTOR CARLOS ANDRES,"[LAGRICEL PF, SOPHIPREN, SOPHIPREN, SOPHIPREN]"
6,"ALARCON CALLUPE,ROSA DE JESUS","[AGGLAD, ELIPTIC PF, LAGRICEL PF, LAGRICEL PF,..."
7,ALCOCER DE LA CRUZ REGINA ROCIO,"[ELIPTIC PF, SOPHIPREN]"
8,ALL VISION E.I.R.L.,"[AGGLAD, ELIPTIC PF, FLUMETOL NF, LAGRICEL PF,..."
9,ANGELES DE LA SALUD,"[AGGLAD, ELIPTIC PF, GAAP PF, TRAZIDEX O, LAGR..."


In [13]:
os.makedirs(f'{BASE}/intermediate', exist_ok=True)
compras.to_csv(f'{BASE}/intermediate/compras_clean.csv', index=False, encoding='utf-8')
seq_por_cliente.to_csv(f'{BASE}/intermediate/secuencias_por_cliente.csv', index=False, encoding='utf-8')

f'Guardado en {BASE}/intermediate'


'Guardado en C:\\Users\\leoro\\Downloads\\TallerIntegradorPruebas/intermediate'

¿Qué contiene cada CSV?
1) compras_clean.csv

Una fila por compra efectiva (solo registros con cantidad > 0).
Te deja todo listo para unir contexto y construir secuencias.

* Columnas típicas (pueden variar según tu BD):

* vendedor, cliente, producto

* mes_num (mes como número), mes_abbr (abreviado, ej. ENE)

* monto_cancelado (dinero pagado ese periodo)

* cantidad (unidades entregadas ese periodo)

* zona_code (Z1, N2, S, …), zona (LIMA, TRUJILLO, CADENAS, …)

Banderas de negocio:

* is_return (devolución/ajuste)

* is_payment_only (solo pago, sin entrega en ese mes)

* is_purchase (TRUE aquí, porque filtramos cantidad>0)

* is_credit_delivery (entrega sin pago ese mes)

Está ordenado por cliente, mes_num, producto.
Usaremos este archivo para unir con: maestra.csv y los resúmenes mensuales.

2) secuencias_por_cliente.csv

Una fila por cliente, con su lista ordenada de productos comprados en el tiempo.

Columnas:

* cliente

* secuencia_productos → una lista tipo ['FLUMETOL NF', 'TRAZIDEX U', 'ELIPTIC PF', ...]

Esto es para visualizar el historial y verificar que la secuencia tenga sentido.
Más adelante, a partir de estas secuencias haremos los pares (historial → siguiente producto) para entrenar la GRU/LSTM.

# **PARTE 2**

**Cargar intermedio de compras y los 3 CSV de contexto**

In [14]:
import pandas as pd, numpy as np


# Intermedio de la Parte 1 (solo compras efectivas: cantidad>0)
compras = pd.read_csv(f'{BASE}/intermediate/compras_clean.csv')

# Resúmenes (vienen con doble cabecera y separador ;)
rprod_raw = pd.read_csv(f'{BASE}/Resumen_Valores-VENTA_POR_PRODUCTO2.csv',
                        sep=';', encoding='latin-1', header=[0,1], dtype=str)
rfam_raw  = pd.read_csv(f'{BASE}/Resumen_Valores-VENTA_POR_FAMILIA2.csv',
                        sep=';', encoding='latin-1', header=[0,1], dtype=str)

# Maestra de productos
maestra   = pd.read_csv(f'{BASE}/maestra.csv', sep=';', encoding='latin-1', dtype=str)

(compras.shape, rprod_raw.shape, rfam_raw.shape, maestra.shape)


((4717, 13), (17, 53), (13, 53), (16, 3))

In [15]:
# Normaliza cabeceras a MultiIndex (MES, METRICA)
lvl0 = [str(a).strip().upper() for a,b in rprod_raw.columns]
lvl1 = [str(b).strip().upper() for a,b in rprod_raw.columns]

lvl0 = [np.nan if 'UNNAMED' in x else x for x in lvl0]
for i in range(len(lvl0)):
    if i>0 and (pd.isna(lvl0[i]) or lvl0[i]=='' or lvl0[i]=='NAN'):
        lvl0[i] = lvl0[i-1]

lvl1 = [x.replace('PY 24','PY24').replace('PY  24','PY24').replace('PY24 ','PY24') for x in lvl1]
lvl1 = ['VENTA' if x.startswith('VENTA') else
        'TGT'   if 'TGT' in x else
        'PY24'  if 'PY24' in x else
        'PCT'   if x in ['%','PCT'] else x for x in lvl1]

rprod_raw.columns = pd.MultiIndex.from_arrays([lvl0, lvl1], names=['MES','METRICA'])

# Localiza columna "Producto" (en cualquiera de los 2 niveles); si no, usa la primera
prod_col = None
for col in rprod_raw.columns:
    if 'PROD' in str(col[0]).upper() or 'PROD' in str(col[1]).upper():
        prod_col = col; break
if prod_col is None:
    prod_col = rprod_raw.columns[0]

producto_series = rprod_raw[prod_col].astype(str).str.strip().str.upper()

# Trabaja solo con columnas (MES, METRICA)
df = rprod_raw.drop(columns=[prod_col]).copy()
df.columns = df.columns.set_names(['mes_nombre','metrica'])

# Índice = producto → apilar (mes, métrica) → ancho con pivot
df.index = producto_series
tmp = df.stack(level=[0,1]).reset_index()
tmp.columns = ['producto', 'mes_nombre', 'metrica', 'valor']

rprod_long = (tmp
              .pivot_table(index=['producto','mes_nombre'],
                           columns='metrica', values='valor', aggfunc='first')
              .reset_index())
rprod_long.columns.name = None
rprod_long = rprod_long.rename(columns={'VENTA':'venta','TGT':'tgt','PY24':'py24','PCT':'pct'})

# Números (coma → punto)
for c in ['venta','tgt','py24','pct']:
    rprod_long[c] = pd.to_numeric(rprod_long[c].astype(str).str.replace(',','.'), errors='coerce')

print(rprod_long.shape)
rprod_long.head(8)


(221, 6)


,producto,mes_nombre,pct,py24,tgt,venta
0,AGGLAD,ABRIL,0.757043,24635.46,120891.240700,91519.90
1,AGGLAD,AGOSTO,-1.000000,25769.42,2560.845455,0.00
2,AGGLAD,DICIEMBRE,-1.000000,71802.00,2681.545455,0.00
3,AGGLAD,ENERO,1.000000,22534.35,110094.810000,110094.81
4,AGGLAD,FEBRERO,0.967162,31785.06,114546.136400,110784.62
5,AGGLAD,JULIO,-0.863449,29784.66,133581.449200,18240.65
6,AGGLAD,JUNIO,-0.235302,20271.95,120891.240700,92445.30
7,AGGLAD,MARZO,0.708034,31785.06,114546.136400,81102.53


Qué es: el resumen por PRODUCTO y MES ya “desanchado”.
Cada fila = (producto, mes).
Columnas:

* producto: nombre del producto (en mayúsculas, limpio).

* mes_nombre: ENERO, FEBRERO, … (y a veces “YTD JUL” si existe).

* venta: monto vendido de ese producto en ese mes (a nivel global, no de un solo cliente).

* tgt: meta de ventas de ese producto en ese mes.

* py24: lo que se logró el año pasado en ese mismo mes (comparativo).

* pct: porcentaje de meta alcanzado ese mes (aprox. 0.75 = 75%).

   * Si se ve 1.0 ⇒ cumplieron la meta.

   * Si se ve >1.0 ⇒ superaron la meta.

   * Si se ve 0.00 ⇒ no avanzaron.

   * Si se ve-1.000000 ⇒ suele significar “sin dato/meta” en el archivo original.

Para qué sirve: es contexto temporal por producto. Luego se lo pegamos a cada compra para que la RNN “sienta” estacionalidad y presión de metas.*

In [16]:
lvl0 = [str(a).strip().upper() for a,b in rfam_raw.columns]
lvl1 = [str(b).strip().upper() for a,b in rfam_raw.columns]

lvl0 = [np.nan if 'UNNAMED' in x else x for x in lvl0]
for i in range(len(lvl0)):
    if i>0 and (pd.isna(lvl0[i]) or lvl0[i]=='' or lvl0[i]=='NAN'):
        lvl0[i] = lvl0[i-1]

lvl1 = [x.replace('PY 24','PY24').replace('PY  24','PY24').replace('PY24 ','PY24') for x in lvl1]
lvl1 = ['VENTA' if x.startswith('VENTA') else
        'TGT'   if 'TGT' in x else
        'PY24'  if 'PY24' in x else
        'PCT'   if x in ['%','PCT'] else x for x in lvl1]

rfam_raw.columns = pd.MultiIndex.from_arrays([lvl0, lvl1], names=['MES','METRICA'])

# Localiza columna Familia (o primera si no la halla)
fam_col = None
for col in rfam_raw.columns:
    if 'FAM' in str(col[0]).upper() or 'FAM' in str(col[1]).upper() or 'FAMILIA' in str(col[0]).upper() or 'FAMILIA' in str(col[1]).upper():
        fam_col = col; break
if fam_col is None:
    fam_col = rfam_raw.columns[0]

familia_series = rfam_raw[fam_col].astype(str).str.strip().str.upper()

df_f = rfam_raw.drop(columns=[fam_col]).copy()
df_f.columns = df_f.columns.set_names(['mes_nombre','metrica'])
df_f.index = familia_series

tmpf = df_f.stack(level=[0,1]).reset_index()
tmpf.columns = ['familia', 'mes_nombre', 'metrica', 'valor']

rfam_long = (tmpf
             .pivot_table(index=['familia','mes_nombre'],
                          columns='metrica', values='valor', aggfunc='first')
             .reset_index())
rfam_long.columns.name = None
rfam_long = rfam_long.rename(columns={'VENTA':'venta_f','TGT':'tgt_f','PY24':'py24_f','PCT':'pct_f'})

for c in ['venta_f','tgt_f','py24_f','pct_f']:
    rfam_long[c] = pd.to_numeric(rfam_long[c].astype(str).str.replace(',','.'), errors='coerce')

print(rfam_long.shape)
rfam_long.head(8)


(156, 6)


,familia,mes_nombre,pct_f,py24_f,tgt_f,venta_f
0,AGGLAD,ABRIL,0.757043,24635.46,120891.240700,91519.90
1,AGGLAD,AGOSTO,0.000000,25769.42,2560.845455,0.00
2,AGGLAD,DICIEMBRE,0.000000,71802.00,2681.545455,0.00
3,AGGLAD,ENERO,1.000000,22534.35,110094.810000,110094.81
4,AGGLAD,FEBRERO,0.967162,31785.06,114546.136400,110784.62
5,AGGLAD,JULIO,0.136551,29784.66,133581.449200,18240.65
6,AGGLAD,JUNIO,0.764698,20271.95,120891.240700,92445.30
7,AGGLAD,MARZO,0.708034,31785.06,114546.136400,81102.53


Qué es: el mismo resumen, pero agregado por FAMILIA y MES.
Cada fila = (familia, mes).
Columnas:

* familia, mes_nombre (igual idea).

* venta_f, tgt_f, py24_f, pct_f (las mismas métricas, pero a nivel familia).

Para qué sirve: cuando un producto tiene poca historia, el comportamiento de su familia ese mes aporta señal.
Más adelante, al tener un mapeo producto → familia, pegaremos estos campos a las compras.

In [17]:
# Maestra → normaliza y renombra
maestra.columns = [c.strip().upper() for c in maestra.columns]
maestra = maestra.rename(columns={'PRODUCTO':'producto','NUMERO DE ARTICULO':'sku','DESCRIPCION':'descripcion'})
maestra['producto'] = maestra['producto'].astype(str).str.strip().str.upper()

compras['producto'] = compras['producto'].astype(str).str.strip().str.upper()
compras = compras.merge(maestra[['producto','sku','descripcion']], how='left', on='producto')

print("Productos sin match en maestra:", compras['sku'].isna().sum())
compras[['producto','sku','descripcion']].head(5)


Productos sin match en maestra: 0


,producto,sku,descripcion
0,GAAP PF,41567,GAAP OFTENO LIBRE DE CONSER PF 3 ML PERU
1,LAGRICEL PF,41582,LAGRICEL OFTENO LIBRE DE CONSERVADORES (PF) 10 ML
2,SOPHIPREN,40338,SOPHIPREN OFTENO 5 ML
3,ZEBESTEN,41604,ZEBESTEN 5ML PERU
4,LAGRICEL PF,41582,LAGRICEL OFTENO LIBRE DE CONSERVADORES (PF) 10 ML


In [18]:
# Mes abreviado → nombre de mes, y merge con contexto por PRODUCTO
MAP_MES = {'ENE':'ENERO','FEB':'FEBRERO','MAR':'MARZO','ABR':'ABRIL','MAY':'MAYO','JUN':'JUNIO',
           'JUL':'JULIO','AGO':'AGOSTO','SET':'SEPTIEMBRE','SEP':'SEPTIEMBRE',
           'OCT':'OCTUBRE','NOV':'NOVIEMBRE','DIC':'DICIEMBRE'}

compras['mes_abbr']   = compras['mes_abbr'].astype(str).str.upper()
compras['mes_nombre'] = compras['mes_abbr'].map(MAP_MES)

compras_ctx = compras.merge(rprod_long, how='left', on=['producto','mes_nombre'])
compras_ctx[['cliente','producto','mes_num','mes_nombre','cantidad','venta','tgt','py24','pct']].head(10)


,cliente,producto,mes_num,mes_nombre,cantidad,venta,tgt,py24,pct
0,A & E SERVICIOS MEDICOS S.A.C.,GAAP PF,1,ENERO,6,99013.28,99013.28000,67726.30,1.000000
1,A & E SERVICIOS MEDICOS S.A.C.,LAGRICEL PF,1,ENERO,6,244552.30,244552.30000,241357.65,1.000000
2,A & E SERVICIOS MEDICOS S.A.C.,SOPHIPREN,1,ENERO,6,50525.90,50525.90000,41450.28,1.000000
3,A & E SERVICIOS MEDICOS S.A.C.,ZEBESTEN,1,ENERO,6,43862.02,43862.02000,31496.43,1.000000
4,A & E SERVICIOS MEDICOS S.A.C.,LAGRICEL PF,2,FEBRERO,6,252555.63,248182.49060,244371.43,1.017621
5,A & E SERVICIOS MEDICOS S.A.C.,SOPHIPREN,2,FEBRERO,12,73566.85,56555.62783,39394.05,1.300787
6,A & E SERVICIOS MEDICOS S.A.C.,ZEBESTEN,2,FEBRERO,12,27431.06,27459.43408,24271.56,0.998967
7,A & E SERVICIOS MEDICOS S.A.C.,ELIPTIC PF,3,MARZO,12,38436.64,70691.40098,16081.78,0.543724
8,A & E SERVICIOS MEDICOS S.A.C.,LAGRICEL PF,3,MARZO,12,258500.41,248182.49060,244371.43,1.041574
9,A & E SERVICIOS MEDICOS S.A.C.,SOPHIPREN,3,MARZO,12,57583.91,56555.62783,39394.05,1.018182


In [19]:
compras_ctx = compras_ctx.drop(columns=['familia','venta_f','tgt_f','py24_f','pct_f'], errors='ignore')


In [20]:
compras_ctx = compras_ctx.sort_values(['cliente','mes_num','producto'])
compras_ctx['cliente_id']  = compras_ctx['cliente'].astype('category').cat.codes
compras_ctx['producto_id'] = compras_ctx['producto'].astype('category').cat.codes

import os
os.makedirs(f'{BASE}/intermediate', exist_ok=True)
compras_ctx.to_csv(f'{BASE}/intermediate/compras_ctx.csv', index=False, encoding='utf-8')

# ¿Quedó todo con contexto por PRODUCTO?
sin_ctx = compras_ctx[compras_ctx['venta'].isna()][['producto','mes_nombre']].drop_duplicates()
print("Compras SIN contexto por producto/mes:", len(sin_ctx))
sin_ctx.head(10)


Compras SIN contexto por producto/mes: 0


,producto,mes_nombre


In [21]:
# Check 1: forma y primeras columnas
compras_ctx.shape, compras_ctx.columns.tolist()[:20]


((4717, 22),
 ['vendedor',
  'cliente',
  'producto',
  'mes_num',
  'mes_abbr',
  'monto_cancelado',
  'cantidad',
  'zona_code',
  'zona',
  'is_return',
  'is_payment_only',
  'is_purchase',
  'is_credit_delivery',
  'sku',
  'descripcion',
  'mes_nombre',
  'pct',
  'py24',
  'tgt',
  'venta'])

In [22]:
# Check 2: una secuencia por cliente (lo que verá la GRU)
(compras_ctx
 .sort_values(['cliente','mes_num'])
 .groupby('cliente')['producto']
 .apply(list)
 .head(5))


cliente
A & E SERVICIOS MEDICOS S.A.C.               [GAAP PF, LAGRICEL PF, SOPHIPREN, ZEBESTEN, LA...
AAA SAC                                      [DUSTALOX, ELIPTIC PF, TRAZIDEX U, ZEBESTEN, E...
ADMINISTRADORA CLINICA RICARDO PALMA S.A.    [AGGLAD, ELIPTIC, FLUMETOL NF, LAGRICEL, SOPHI...
ADMINISTRADORA CLINICA TRESA S.A             [AGGLAD, FLUMETOL NF, GAAP, LAGRICEL, TRAZIDEX...
AGUIRRE ROCCA CESAR JOAQUIN                                                       [TRAZIDEX O]
Name: producto, dtype: object

In [23]:
# Check 3: % de NaN en las 4 columnas de contexto por producto
compras_ctx[['venta','tgt','py24','pct']].isna().mean()


venta    0.0
tgt      0.0
py24     0.0
pct      0.0
dtype: float64

# **PARTE 3**

In [24]:
import pandas as pd, numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

compras_ctx = pd.read_csv(f'{BASE}/intermediate/compras_ctx.csv')

# Asegurar tipos
compras_ctx['cliente_id']  = compras_ctx['cliente_id'].astype(int)
compras_ctx['producto_id'] = compras_ctx['producto_id'].astype(int)
compras_ctx['mes_num']     = compras_ctx['mes_num'].astype(int)

# Orden temporal por cliente
compras_ctx = compras_ctx.sort_values(['cliente_id','mes_num'])
compras_ctx.head(5)


,vendedor,cliente,producto,mes_num,mes_abbr,monto_cancelado,cantidad,zona_code,zona,is_return,is_payment_only,is_purchase,is_credit_delivery,sku,descripcion,mes_nombre,pct,py24,tgt,venta,cliente_id,producto_id
0,PHARMA - Z1,A & E SERVICIOS MEDICOS S.A.C.,GAAP PF,1,ENE,540.93,6,Z1,LIMA,False,False,True,False,41567,GAAP OFTENO LIBRE DE CONSER PF 3 ML PERU,ENERO,1.000000,67726.30,99013.2800,99013.28,0,8
1,PHARMA - Z1,A & E SERVICIOS MEDICOS S.A.C.,LAGRICEL PF,1,ENE,340.83,6,Z1,LIMA,False,False,True,False,41582,LAGRICEL OFTENO LIBRE DE CONSERVADORES (PF) 10 ML,ENERO,1.000000,241357.65,244552.3000,244552.30,0,10
2,PHARMA - Z1,A & E SERVICIOS MEDICOS S.A.C.,SOPHIPREN,1,ENE,254.10,6,Z1,LIMA,False,False,True,False,40338,SOPHIPREN OFTENO 5 ML,ENERO,1.000000,41450.28,50525.9000,50525.90,0,12
3,PHARMA - Z1,A & E SERVICIOS MEDICOS S.A.C.,ZEBESTEN,1,ENE,294.53,6,Z1,LIMA,False,False,True,False,41604,ZEBESTEN 5ML PERU,ENERO,1.000000,31496.43,43862.0200,43862.02,0,15
4,PHARMA - Z1,A & E SERVICIOS MEDICOS S.A.C.,LAGRICEL PF,2,FEB,340.83,6,Z1,LIMA,False,False,True,False,41582,LAGRICEL OFTENO LIBRE DE CONSERVADORES (PF) 10 ML,FEBRERO,1.017621,244371.43,248182.4906,252555.63,0,10


**Construir pares (historial → siguiente producto)**

Idea simple: por cliente, ordenamos por mes_num y creamos ventanas:

* Si el historial del cliente es [p1, p2, p3], generamos:

   * input=[p1] → target=p2

   * input=[p1, p2] → target=p3

Usaremos un largo máximo (MAXLEN) y padding a la izquierda con 0.

Nota: para reservar el 0 como PAD, desplazamos todos los producto_id en +1.

Se acaba de crear:

X_*: matrices de tamaño [n_ejemplos, MAXLEN] con enteros (producto_id + 1) y 0s como PAD.

y_*: vectores con el próximo producto (también con offset +1).

Convención de padding: reservamos el id=0 para PAD. Por eso desplazamos los productos reales a id+1.

In [25]:
# Desplazar los producto_id para reservar 0 como PAD
compras_ctx['prod_pad'] = compras_ctx['producto_id'] + 1
num_items = compras_ctx['prod_pad'].max() + 1  # incluye PAD=0
num_items


np.int64(17)

In [26]:
# Secuencias por cliente (lista de productos) y, además, contexto por paso (venta, tgt, py24, pct)
cols_ctx = ['venta','tgt','py24','pct']
X_seqs, X_ctx_seqs, y_next = [], [], []
Xv_seqs, Xv_ctx_seqs, yv_next = [], [], []

for cid, df in compras_ctx.groupby('cliente_id'):
    items = df['prod_pad'].tolist()
    ctxs  = df[cols_ctx].values.astype('float32')  # [pasos, 4]

    if len(items) < 2:
        continue

    # pares de TRAIN: todos menos el último salto
    for t in range(1, len(items)-1):
        X_seqs.append(items[:t])         # historial hasta t-1
        X_ctx_seqs.append(ctxs[:t])      # contexto de esos pasos
        y_next.append(items[t])          # siguiente producto en t

    # par de VALID: solo el último salto
    Xv_seqs.append(items[:-1])
    Xv_ctx_seqs.append(ctxs[:-1])
    yv_next.append(items[-1])

len(X_seqs), len(Xv_seqs)


(4042, 322)

X_seqs y y_next son train.

Xv_seqs y yv_next son validación (el último paso de cada cliente).

Padding (longitudes variables → tensores fijos)

* Limitamos el largo máximo que mirará la GRU (recomendado 12).

* Padding a la izquierda con 0 (PAD). Para el contexto, rellenamos con ceros.

In [27]:
MAX_LEN = 12

# TRAIN
X_pad = np.zeros((len(X_seqs), MAX_LEN), dtype='int64')
C_pad = np.zeros((len(X_seqs), MAX_LEN, len(cols_ctx)), dtype='float32')
L_len = np.zeros((len(X_seqs),), dtype='int64')
y_arr = np.array(y_next, dtype='int64') - 1  # target en [0..num_items-2] (quitamos PAD)

for i, (seq, cseq) in enumerate(zip(X_seqs, X_ctx_seqs)):
    s = seq[-MAX_LEN:]
    c = cseq[-MAX_LEN:]
    X_pad[i, -len(s):] = s
    C_pad[i, -len(s):, :] = c
    L_len[i] = len(s)

# VALID
Xv_pad = np.zeros((len(Xv_seqs), MAX_LEN), dtype='int64')
Cv_pad = np.zeros((len(Xv_seqs), MAX_LEN, len(cols_ctx)), dtype='float32')
Lv_len = np.zeros((len(Xv_seqs),), dtype='int64')
yv_arr = np.array(yv_next, dtype='int64') - 1

for i, (seq, cseq) in enumerate(zip(Xv_seqs, Xv_ctx_seqs)):
    s = seq[-MAX_LEN:]
    c = cseq[-MAX_LEN:]
    Xv_pad[i, -len(s):] = s
    Cv_pad[i, -len(s):, :] = c
    Lv_len[i] = len(s)

X_pad.shape, C_pad.shape, y_arr.shape, Xv_pad.shape, yv_arr.shape


((4042, 12), (4042, 12, 4), (4042,), (322, 12), (322,))

**Tensores y device**

In [28]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_pad_t  = torch.tensor(X_pad, dtype=torch.long, device=device)
C_pad_t  = torch.tensor(C_pad, dtype=torch.float32, device=device)
L_len_t  = torch.tensor(L_len, dtype=torch.long, device=device)
y_arr_t  = torch.tensor(y_arr, dtype=torch.long, device=device)

Xv_pad_t = torch.tensor(Xv_pad, dtype=torch.long, device=device)
Cv_pad_t = torch.tensor(Cv_pad, dtype=torch.float32, device=device)
Lv_len_t = torch.tensor(Lv_len, dtype=torch.long, device=device)
yv_arr_t = torch.tensor(yv_arr, dtype=torch.long, device=device)

num_items_no_pad = num_items - 1  # para la clasificación (quitamos el PAD)


# **GRU baseline (sin contexto)**

In [30]:
# ==============================================================================
# PASO 8: DEFINICIÓN DEL MODELO GRU (para Next Item Prediction con Contexto)
# ==============================================================================

class NextItemGRU(nn.Module):
    def __init__(self, num_items, emb_dim=64, hidden=128, num_layers=1, dropout=0.0, ctx_features_dim=4):
        super(NextItemGRU, self).__init__()

        # Embedding del Item (para los IDs de productos)
        # num_items incluye el PAD (0) y los num_items_no_pad reales
        self.item_emb = nn.Embedding(num_items, emb_dim, padding_idx=0)

        # GRU Layer
        # Input size = dimensión del embedding del item + dimensión de los features de contexto
        self.gru = nn.GRU(
            input_size=emb_dim + ctx_features_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        # Capa de salida para clasificación (predice el siguiente item)
        # num_items - 1 porque no queremos predecir el token PAD como un item real
        self.output_layer = nn.Linear(hidden, num_items - 1)

    def forward(self, item_sequences, context_sequences, lengths):
        # item_sequences: [Batch, MaxLen] - IDs de productos (padded con 0)
        # context_sequences: [Batch, MaxLen, CtxDim] - Features de contexto por paso
        # lengths: [Batch] - Longitudes reales de las secuencias

        # Obtener embeddings de los items
        item_embs = self.item_emb(item_sequences) # [Batch, MaxLen, emb_dim]

        # Concatenar embeddings de items con features de contexto
        combined_input = torch.cat([item_embs, context_sequences], dim=2) # [Batch, MaxLen, emb_dim + CtxDim]

        # Empaquetar secuencias para la GRU (manejo eficiente de longitudes variables)
        # Permite que GRU solo procese los elementos reales, ignorando el padding
        packed_input = nn.utils.rnn.pack_padded_sequence(combined_input, lengths.cpu(), batch_first=True, enforce_sorted=False)

        # Pasar por GRU
        packed_output, h_n = self.gru(packed_input)

        # h_n es el estado oculto del último paso de las secuencias reales
        # h_n shape: [num_layers, Batch, hidden]
        # Tomamos el último estado de la última capa (si num_layers > 1)
        last_hidden_state = h_n[-1] # [Batch, hidden]

        # La capa de salida predice logits para cada posible item (excluyendo PAD)
        logits = self.output_layer(last_hidden_state) # [Batch, num_items - 1]

        return logits

# Configuración e inicialización del modelo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ctx_features_dim es la longitud de cols_ctx, que es 4
model = NextItemGRU(num_items=num_items, emb_dim=64, hidden=128, num_layers=1, dropout=0.0, ctx_features_dim=len(cols_ctx)).to(device)

print("Modelo NextItemGRU Inicializado en:", device)


Modelo NextItemGRU Inicializado en: cpu


In [ ]:
# ==============================================================================
# PASO 8: DEFINICIÓN DEL MODELO GRU (Adaptado de tu código)
# ==============================================================================

class SalesGRU(nn.Module):
    def __init__(self, num_items, emb_dim=64, hidden=128, num_layers=1, dropout=0.0, input_features=3):
        super(SalesGRU, self).__init__()

        # 1. Embedding del Item (Tal como pediste)
        # Aprende una representación vectorial de cada producto
        self.item_emb = nn.Embedding(num_items, emb_dim, padding_idx=0)

        # 2. GRU
        # Input size = Features numéricos (Ventas, Precio...) + Dimensión del Embedding
        self.gru = nn.GRU(
            input_size=input_features + emb_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        # 3. Capas de Salida (Regresión)
        self.proj_he = nn.Linear(hidden, 64)
        self.output = nn.Linear(64, 1) # Predice 1 valor: Venta_Cajas
        self.relu = nn.ReLU()

    def forward(self, x_seq, item_idx):
        # x_seq: [Batch, Time, Features] -> Datos dinámicos (Ventas históricas)
        # item_idx: [Batch] -> Dato estático (Qué producto es)

        # A. Obtener Embedding del Producto y repetirlo para cada paso de tiempo
        emb = self.item_emb(item_idx) # [Batch, emb_dim]
        emb_expanded = emb.unsqueeze(1).repeat(1, x_seq.size(1), 1) # [Batch, Time, emb_dim]

        # B. Concatenar Embedding con Features Numéricos
        # Ahora la GRU sabe "Cuánto se vendió" Y "Qué producto es"
        combined_input = torch.cat([x_seq, emb_expanded], dim=2)

        # C. Pasar por GRU
        out, h = self.gru(combined_input)
        h_last = h[-1] # Último estado oculto

        # D. Predicción
        x = self.relu(self.proj_he(h_last))
        pred = self.output(x) # Salida lineal (porque es regresión)
        return pred

# Configuración
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SalesGRU(num_items=num_items, emb_dim=64, hidden=128, input_features=3).to(device)

print("Modelo GRU Inicializado en:", device)

In [31]:
# ==============================================================================
# PASO 9: ENTRENAMIENTO (Training Loop)
# ==============================================================================

# Hiperparámetros
BATCH = 64
EPOCHS = 15
lr = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

# Preparar DataLoader
dataset = torch.utils.data.TensorDataset(X_pad_t, C_pad_t, L_len_t, y_arr_t)
loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH, shuffle=True)

print(f"--- Iniciando Entrenamiento ({EPOCHS} Epochs) ---")
model.train()

for epoch in range(1, EPOCHS+1):
    total_loss = 0.0
    steps = 0

    for batch_x, batch_c, batch_l, batch_y in loader:
        batch_x, batch_c, batch_l, batch_y = batch_x.to(device), batch_c.to(device), batch_l.to(device), batch_y.to(device)

        preds = model(batch_x, batch_c, batch_l)
        loss = criterion(preds, batch_y)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total_loss += loss.item()
        steps += 1

    avg_loss = total_loss / steps
    print(f'Epoch {epoch:02d} | Loss (CrossEntropy): {avg_loss:.4f}')

print("Entrenamiento completado.")

--- Iniciando Entrenamiento (15 Epochs) ---
Epoch 01 | Loss (CrossEntropy): 2.6658
Epoch 02 | Loss (CrossEntropy): 2.6075
Epoch 03 | Loss (CrossEntropy): 2.5862
Epoch 04 | Loss (CrossEntropy): 2.5544
Epoch 05 | Loss (CrossEntropy): 2.5261
Epoch 06 | Loss (CrossEntropy): 2.5173
Epoch 07 | Loss (CrossEntropy): 2.5104
Epoch 08 | Loss (CrossEntropy): 2.4974
Epoch 09 | Loss (CrossEntropy): 2.5039
Epoch 10 | Loss (CrossEntropy): 2.4978
Epoch 11 | Loss (CrossEntropy): 2.4894
Epoch 12 | Loss (CrossEntropy): 2.5090
Epoch 13 | Loss (CrossEntropy): 2.4929
Epoch 14 | Loss (CrossEntropy): 2.4935
Epoch 15 | Loss (CrossEntropy): 2.4916
Entrenamiento completado.


Interpretación: compara los valores con los que obtengas en el modelo con contexto (abajo). Lo normal es que Recall@10 esté por encima de Recall@5, y que NDCG sea más bajo que Recall (porque pondera la posición).

# REFERENCIAS

Alotaibi, F. M. (2023). A Machine-Learning-Inspired Opinion Extraction Mechanism for Classifying Customer Reviews on Social Media. Applied Sciences, 13(12), 7266. https://doi.org/10.3390/app13127266

Bagwari, A., Sinha, A., Singh, N. K., Garg, N., & Kanti, J. (2022). CBIR-DSS: Business Decision Oriented Content-Based Recommendation Model for E-Commerce. Information, 13(10), 479. https://doi.org/10.3390/info13100479
Bilal, A. I., Bititci, U. S., & Fenta, T. G. (2024). Challenges and the Way Forward in Demand-Forecasting Practices within the Ethiopian Public Pharmaceutical Supply Chain. Pharmacy, 12(3), 86. https://doi.org/10.3390/pharmacy12030086

Islam, M. M., & Baek, J.-H. (2021). Deep Learning Based Real Age and Gender Estimation from Unconstrained Face Image towards Smart Store Customer Relationship Management. Applied Sciences, 11(10), 4549. https://doi.org/10.3390/app11104549

Lee, M., & Kim, H.-J. (2023). A collaborative filtering model incorporating media promotions and users’ variety-seeking tendencies in the digital music market. Decision Support Systems, 174, 114022. https://doi.org/10.1016/j.dss.2023.114022

Maher, M., Ngoy, P. M., Rebriks, A., Ozcinar, C., Cuevas, J., Sanagavarapu, R., & Anbarjafari, G. (2022). Comprehensive Empirical Evaluation of Deep Learning Approaches for Session-Based Recommendation in E-Commerce. Entropy, 24(11), 1575. https://doi.org/10.3390/e24111575

Pandey, A., Mannepalli, P. K., Gupta, M., Dangi, R., & Choudhary, G. (2024). A Deep Learning-Based Hybrid CNN-LSTM Model for Location-Aware Web Service Recommendation. Neural Processing Letters, 56(5), 234. https://doi.org/10.1007/s11063-024-11687-w

Qian, F., Chen, W., Chen, H., Liu, J., Zhao, S., & Zhang, Y. (2025). Building robust deep recommender systems: Utilizing a weighted adversarial noise propagation framework with robust fine-tuning modules. Knowledge-Based Systems, 314, 113181. https://doi.org/10.1016/j.knosys.2025.113181
Sun, Z., Harit, A., Yu, J., Wang, J., & Liò, P. (2025). Advanced Hypergraph Mining for Web Applications Using Sphere Neural Networks. Companion Proceedings of the ACM on Web Conference 2025, 1316-1320. https://doi.org/10.1145/3701716.3715577

Tholib, A., Widiyaningtyas, T., & Prasetya, D. D. (2025). An Intelligent Recommendation System Utilizing a Hybrid Deep Learning Method. Engineering, Technology & Applied Science Research, 15(4), 25971-25977. https://doi.org/10.48084/etasr.12230

Xiao, M., Zhou, Q., Lu, L., Tao, X., He, W., & Zhou, Y. (2022). Applying Deep Learning-Based Personalized Item Recommendation for Mobile Service in Retailor Industry. Mobile Information Systems, 2022(1), 2364154. https://doi.org/10.1155/2022/2364154

Zhang, S., Yao, L., Sun, A., & Tay, Y. (2019). Deep Learning Based Recommender System: A Survey and New Perspectives. ACM Comput. Surv., 52(1), 5:1-5:38. https://doi.org/10.1145/3285029

Zhang, X., Guo, F., Chen, T., Pan, L., Beliakov, G., & Wu, J. (2023). A Brief Survey of Machine Learning and Deep Learning Techniques for E-Commerce Research. Journal of Theoretical and Applied Electronic Commerce Research, 18(4), 2188-2216. https://doi.org/10.3390/jtaer18040110



